# 13 — Init Signal Debug: does the SALT init carry ANY crosslingual signal?

Context: matched 73-step CPT gave random **5.578** < v5 SALT **5.840** — random won. The training
code is verified (arms loaded distinct weights, same data/eval); the open question is **init tensor
quality**: do v5's projected embeddings encode vi→en semantics at all? Prior forensics said
probably not (projection kNN preservation 0.026 ≈ chance; ViDeBERTa static-table soundness ≈ chance).

This notebook decides it without training:
1. **A/B hygiene** — fingerprint both artifacts' embeddings (proves the CPT runs compared different tensors).
2. **Translation-retrieval probe** — for ~100 common NON-anchor Vietnamese words, rank the true
   English NeoBERT row by cosine among all 30K rows. Anchors are excluded (verbatim copies = trivial).
3. **Step-0 loss decomposition** — full / bias-only / weights-only (CPU forward — L4 CUDA NaNs).

| outcome | meaning | next move |
|---|---|---|
| v5 ranks ≈ random floor | init is semantic noise → today's CPT result fully explained | donor/projection fix: PhoBERT donor, mean-centering, Procrustes rung |
| v5 ranks ≪ floor (e.g. median < 1000, acc@10 > 5%) | signal exists but training can't use it | utilization: decoder scale, freeze-train phase, longer budget |

In [1]:
%%capture
!pip install -U transformers safetensors huggingface_hub sentencepiece pandas

In [2]:
import sys, json, hashlib
from pathlib import Path
import torch
import torch.nn.functional as F

try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped:', e)
PROJECT_ROOT = Path('/content/drive/MyDrive/SALT3')
sys.path.insert(0, str(PROJECT_ROOT / 'code')); sys.path.insert(0, '/content')
import importlib
import salt3_common as sc; importlib.reload(sc)
import salt3_init_signal_probe as probe; importlib.reload(probe)

V5_DIR       = PROJECT_ROOT / 'init' / 'videberta_salt_init_v5_globalmap_freqbias'
BASELINE_DIR = PROJECT_ROOT / 'init' / 'neobert_random_init_baseline'
print('v5      :', V5_DIR)
print('baseline:', BASELINE_DIR)

Mounted at /content/drive
v5      : /content/drive/MyDrive/SALT3/init/videberta_salt_init_v5_globalmap_freqbias
baseline: /content/drive/MyDrive/SALT3/init/neobert_random_init_baseline


## 1. A/B hygiene — were the two CPT runs really training different embeddings?

In [3]:
from safetensors.torch import load_file

def emb_fingerprint(init_dir):
    w = load_file(str(Path(init_dir) / 'model' / 'model.safetensors'))['model.encoder.weight'].float()
    sha = hashlib.sha256(w[:256].contiguous().numpy().tobytes()).hexdigest()[:12]
    return w, {'shape': tuple(w.shape), 'mean_norm': round(float(w.norm(dim=1).mean()), 4), 'sha12': sha}

w_v5, fp_v5 = emb_fingerprint(V5_DIR)
w_bl, fp_bl = emb_fingerprint(BASELINE_DIR)
print('v5      :', fp_v5)
print('baseline:', fp_bl)
assert fp_v5['sha12'] != fp_bl['sha12'], 'SAME embeddings — the A/B was invalid!'
row_cos = F.cosine_similarity(w_v5, w_bl, dim=1)
print(f'row-wise cosine v5 vs baseline: mean {row_cos.mean():.4f} (≈0 expected for unrelated tensors)')
print('✓ artifacts hold different embeddings — the CPT A/B compared real alternatives')

v5      : {'shape': (30522, 768), 'mean_norm': 1.2183, 'sha12': 'c8602bf4b19c'}
baseline: {'shape': (30522, 768), 'mean_norm': 1.2063, 'sha12': 'c981e3c2e78c'}
row-wise cosine v5 vs baseline: mean 0.0006 (≈0 expected for unrelated tensors)
✓ artifacts hold different embeddings — the CPT A/B compared real alternatives


## 2. Translation-retrieval probe (the decisive measurement)

`run_probe` excludes every anchor token (their rows are verbatim NeoBERT copies — including them
would fake signal), ranks each probe word's projected row against all ~30K NeoBERT rows, and
prints an empirical random floor alongside. `device='cpu'` for the loss decomposition because this
runtime class (L4) NaNs on CUDA forward.

In [4]:
res_v5 = probe.run_probe(V5_DIR, device='cpu')

── SALT init signal probe: videberta_salt_init_v5_globalmap_freqbias ──
  decoder_init=global_emb_to_decoder_map anchors=4990 projection_stats={'lstsq': 25457, 'weighted_avg': 70, 'random_fallback': 0}


tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.31k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/981M [00:00<?, ?B/s]

  probes usable=103 skipped: in-anchor-set=1 not-in-vocab=0

── Translation retrieval (non-anchor tokens, cosine rank of gold EN row) ──
  SALT init          n=103 acc@1=10.7% acc@10=40.8% acc@100=67.0% median_rank=16/30522 MRR=0.1980
  random control     n=103 acc@1=0.0% acc@10=0.0% acc@100=0.0% median_rank=12613/30522 MRR=0.0003
  best 10 : [('chợ', 'market', 1), ('cơm', 'rice', 1), ('dạy', 'teach', 1), ('lạnh', 'cold', 1), ('nghe', 'hear', 1), ('nhỏ', 'small', 1), ('sông', 'river', 1), ('tóc', 'hair', 1), ('vườn', 'garden', 1), ('yêu', 'love', 1)]
  worst 10: [('tuyết', 'snow', 17321), ('cầu', 'bridge', 18063), ('sao', 'star', 18293), ('ngồi', 'sit', 20799), ('sắt', 'iron', 24617), ('nước', 'water', 25692), ('ngày', 'day', 26735), ('mắt', 'eye', 29243), ('mới', 'new', 30059), ('lá', 'leaf', 30519)]

── Step-0 loss decomposition ──
  ✓ load_model_safe: all 172 keys loaded successfully
  step-0 MLM loss: full=7.36 | bias_only=7.27 | weights_only=10.20
  (full ≈ bias_only ⇒ health-gate

In [5]:
res_bl = probe.run_probe(BASELINE_DIR, device='cpu')   # control: true floor of the metric

── SALT init signal probe: neobert_random_init_baseline ──
  decoder_init=global_emb_to_decoder_map anchors=0 projection_stats={'lstsq': 25457, 'weighted_avg': 70, 'random_fallback': 0}
  probes usable=104 skipped: in-anchor-set=0 not-in-vocab=0

── Translation retrieval (non-anchor tokens, cosine rank of gold EN row) ──
  SALT init          n=104 acc@1=0.0% acc@10=0.0% acc@100=1.0% median_rank=15078/30522 MRR=0.0004
  random control     n=104 acc@1=0.0% acc@10=0.0% acc@100=0.0% median_rank=13607/30522 MRR=0.0003
  best 10 : [('biển', 'sea', 45), ('nước', 'water', 514), ('cây', 'tree', 577), ('bạc', 'silver', 823), ('mạnh', 'strong', 1949), ('nhanh', 'fast', 2071), ('miệng', 'mouth', 2916), ('sách', 'book', 3276), ('gỗ', 'wood', 3345), ('yêu', 'love', 3425)]
  worst 10: [('cầu', 'bridge', 27955), ('chó', 'dog', 28300), ('rồng', 'dragon', 28345), ('chồng', 'husband', 28476), ('xương', 'bone', 28624), ('mới', 'new', 29048), ('mẹ', 'mother', 29073), ('thấy', 'see', 29959), ('hổ', 'tiger',

## 3. Verdict

In [6]:
s, r = res_v5['salt'], res_v5['random_control']
b = res_bl['salt']   # baseline artifact scored by the same metric = empirical floor
V = 30522

print(f"{'arm':22s} {'acc@10':>8s} {'acc@100':>8s} {'median_rank':>12s} {'MRR':>8s}")
for name, st in (('v5 SALT init', s), ('random control', r), ('baseline artifact', b)):
    print(f"{name:22s} {st['acc10']:8.1%} {st['acc100']:8.1%} {st['median_rank']:12d} {st['mrr']:8.4f}")

ld = res_v5.get('loss_decomposition', {})
if ld:
    print(f"\nv5 loss decomposition: full={ld['full']:.2f} bias_only={ld['bias_only']:.2f} "
          f"weights_only={ld['weights_only']:.2f}")
    print(f"  context+weights contribution at step 0 = {ld['bias_only'] - ld['full']:+.3f} nats")

signal = s['median_rank'] < r['median_rank'] / 5 or s['acc10'] >= 0.05
print('\n' + '=' * 70)
if signal:
    print('VERDICT: v5 embeddings DO carry crosslingual signal that CPT failed to use.')
    print('→ problem = UTILIZATION: test decoder scale arms (01c), embedding freeze-train')
    print('  phase, and longer matched budgets before touching the projection.')
else:
    print('VERDICT: v5 embeddings are at the random floor — semantic NOISE.')
    print('→ today\'s CPT result (random ≥ SALT) is fully explained: there was no signal')
    print('  to use. The code is fine; the INIT CONTENT is the problem.')
    print('→ fix upstream: PhoBERT donor arm, mean-centering before the local fits,')
    print('  Procrustes-from-anchors init. The 01c decoder arms are secondary until then.')
print('=' * 70)

arm                      acc@10  acc@100  median_rank      MRR
v5 SALT init              40.8%    67.0%           16   0.1980
random control             0.0%     0.0%        12613   0.0003
baseline artifact          0.0%     1.0%        15078   0.0004

v5 loss decomposition: full=7.36 bias_only=7.27 weights_only=10.20
  context+weights contribution at step 0 = -0.087 nats

VERDICT: v5 embeddings DO carry crosslingual signal that CPT failed to use.
→ problem = UTILIZATION: test decoder scale arms (01c), embedding freeze-train
  phase, and longer matched budgets before touching the projection.
